In [3]:
import os
from typing import TypedDict
from langchain_core.prompts import PromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.graph import StateGraph, START, END
from dotenv import load_dotenv

load_dotenv()

# 1. Define the Shared State
class BlogState(TypedDict):
    topic: str
    outline: str
    sections: str
    final_draft: str


# 2. Initialize Model
# Make sure GOOGLE_API_KEY is set in your environment
llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite", temperature=0.7)


# 3. Define Node Functions
def generate_outline(state: BlogState) -> dict:
    prompt = PromptTemplate.from_template(
        "You are an expert content strategist. Create a detailed blog outline for the topic: '{topic}'. "
        "Include an introduction, 3 main sections, and a conclusion."
    )
    chain = prompt | llm
    response = chain.invoke({"topic": state["topic"]})
    return {"outline": response.content}


def expand_sections(state: BlogState) -> dict:
    prompt = PromptTemplate.from_template(
        "Based on the following blog outline:\n\n{outline}\n\n"
        "Expand each main section with detailed bullet points, key takeaways, and main sub-arguments."
    )
    chain = prompt | llm
    response = chain.invoke({"outline": state["outline"]})
    return {"sections": response.content}


def write_draft(state: BlogState) -> dict:
    prompt = PromptTemplate.from_template(
        "You are a professional blog writer. Using the blog topic '{topic}', the outline:\n{outline}\n\n"
        "and expanded sections:\n{sections}\n\n"
        "Write a complete, polished, and engaging blog post formatted cleanly in Markdown."
    )
    chain = prompt | llm
    response = chain.invoke({
        "topic": state["topic"],
        "outline": state["outline"],
        "sections": state["sections"]
    })
    return {"final_draft": response.content}


# 4. Build and Assemble the Graph
builder = StateGraph(BlogState)

# Add nodes
builder.add_node("generate_outline", generate_outline)
builder.add_node("expand_sections", expand_sections)
builder.add_node("write_draft", write_draft)

# Connect edges sequentially (Prompt Chaining Flow)
builder.add_edge(START, "generate_outline")
builder.add_edge("generate_outline", "expand_sections")
builder.add_edge("expand_sections", "write_draft")
builder.add_edge("write_draft", END)

# Compile the executable app
app = builder.compile()


# 5. Execution Example
if __name__ == "__main__":
    initial_input = {"topic": "The Future of AI Agents in Quantitative Finance"}
    
    # Run the graph
    result = app.invoke(initial_input)
    
    print("\n--- GENERATED OUTLINE ---\n")
    print(result["outline"])
    
    print("\n--- FINAL BLOG DRAFT ---\n")
    print(result["final_draft"])

d:\anaconda3\envs\lgenv\Lib\site-packages\langchain_google_genai\chat_models.py:3935: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
d:\anaconda3\envs\lgenv\Lib\site-packages\langchain_google_genai\chat_models.py:3935: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
d:\anaconda3\envs\lgenv\Lib\site-packages\langchain_google_genai\chat_models.py:3935: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(



--- GENERATED OUTLINE ---

[{'type': 'text', 'text': 'As an expert content strategist, I have designed this blog outline to target a sophisticated audience of quantitative researchers, financial technologists, institutional investors, and fintech enthusiasts. \n\nThe tone should be authoritative, forward-thinking, and analytical, balancing technical reality with strategic foresight.\n\n---\n\n# Blog Outline: The Future of AI Agents in Quantitative Finance\n\n## **I. Introduction**\n*   **The Hook:** Briefly trace the evolution of quantitative finance—from manual trading pits to rule-based algorithms, and now, to autonomous systems. Mention how traditional "black box" models are giving way to dynamic, goal-oriented architectures.\n*   **Defining the Shift:** Differentiate between traditional machine learning models (which predict outcomes based on static data) and **AI Agents** (which possess perception, reasoning, planning, and execution capabilities).\n*   **Thesis Statement:** AI ag